# T2.2 Semantic Mapping

Owner: B

This notebook maps every column in the DBRepo tables to a concept in a recognised ontology.
This addresses the F (Findable) and I (Interoperable) aspects of FAIR.

We use five ontologies:

- **SOSA** for the observation structure (station, date, quality flags). It is a W3C standard made specifically for sensor-based measurements.
- **CHEBI** for all the chemical species (Pb, Cd, Ca, Mg, Na, K, Cl, NO3, SO4, NH4). ChEBI is the standard chemistry ontology with stable URIs for every ion.
- **ENVO** for precipitation as the environmental phenomenon being measured.
- **PATO** for pH and electrical conductivity, since these are physical qualities that CHEBI does not cover.
- **WGS84** for the station coordinates.

WikiData was not used because all the concepts we need are already in these more specific ontologies.

In [2]:
import os
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv("../.env")
#ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
#USERNAME    = "e12551187@student.tuwien.ac.at"
#PASSWORD    = "@Puthenpurayil1"
#DATABASE_ID = "bfa4385b-54a9-4ae3-b4f4-cb503d7bb016"

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")


client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("logged in as:", client.whoami())

federicoa88
logged in as: federicoa88


In [3]:
# table IDs from Owner A's notebook

TABLE_STATIONS      = "53688cc1-5205-4f25-af30-7feef2ea1b2b"
TABLE_PRECIPITATION = "d11966e6-f0a6-460d-900b-b56e627fc752"
TABLE_VARIABLES     = "7ed509f5-3356-4318-80b3-c6672b13c4b8"

In [4]:
# get the column UUIDs from DBRepo
# update_table_column needs the column UUID, not the column name

def get_column_ids(table_id):
    table = client.get_table(database_id=DATABASE_ID, table_id=table_id)
    return {col.name: col.id for col in table.columns}

stations_col_ids      = get_column_ids(TABLE_STATIONS)
precipitation_col_ids = get_column_ids(TABLE_PRECIPITATION)
variables_col_ids     = get_column_ids(TABLE_VARIABLES)

print("stations columns:")
for name, cid in stations_col_ids.items():
    print(f"  {name:20s} {cid}")

print("\nprecipitation columns:")
for name, cid in precipitation_col_ids.items():
    print(f"  {name:20s} {cid}")

print("\nmeasurement_variables columns:")
for name, cid in variables_col_ids.items():
    print(f"  {name:20s} {cid}")

stations columns:
  station_id           9f762d20-696c-477f-8b74-7f7ba052f579
  station_code         3ed0f861-e3ec-45e4-a48a-16fb8bf3852e
  latitude             2e160049-5b95-4435-b490-8e2e628e4c30
  longitude            0e0d51f7-55d3-4358-b355-dac60f65f8e1

precipitation columns:
  measurement_id       db5c0ce7-5cac-4ac1-84a8-dd707158ec6c
  station_id           97f27df3-c1da-4655-9c2d-9649d82c4e73
  sample_date          d2d0c87a-3c70-4a25-99cd-421fcecc2d65
  NS                   a6f25f3d-58e2-4e98-a6d2-a5bb13b8ad19
  NS_flag              b8bf398f-b885-4d0d-9778-1106de00c248
  LF                   5a44eea2-3486-4aa4-875e-2b0df2d277d4
  LF_flag              9e669b90-dd47-4de2-b63d-2f58ce68a228
  pH                   3d4efc42-5086-4ab4-ab81-4a2fd0979575
  pH_flag              0d74e446-16ae-4c24-b461-316f40442328
  NH4                  2d2e5447-a73c-4781-af73-6587bda25a82
  NH4_flag             dfaa719e-5d07-4b9f-b760-d9c29f4c70fa
  Na                   f01af1d3-a3a4-48e9-95b4-7545147df10

In [10]:
# each row: (table_id, column_ids_dict, column_name, concept_uri, ontology, label)
# internal ID columns (station_id, measurement_id, variable_id) are skipped

mappings = [

    # stations table
    (TABLE_STATIONS, stations_col_ids, "station_code", "https://www.w3.org/ns/sosa/Platform",             "SOSA",  "Platform"),
    (TABLE_STATIONS, stations_col_ids, "latitude",     "http://www.w3.org/2003/01/geo/wgs84_pos#lat",      "WGS84", "latitude"),
    (TABLE_STATIONS, stations_col_ids, "longitude",    "http://www.w3.org/2003/01/geo/wgs84_pos#long",     "WGS84", "longitude"),

    # precipitation_measurements table
    (TABLE_PRECIPITATION, precipitation_col_ids, "sample_date", "https://www.w3.org/ns/sosa/phenomenonTime",   "SOSA",  "phenomenonTime"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "station_id",  "https://www.w3.org/ns/sosa/Platform",         "SOSA",  "Platform"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NS",  "http://purl.obolibrary.org/obo/ENVO_01001783",         "ENVO",  "precipitation measurement"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "LF",  "http://purl.obolibrary.org/obo/PATO_0001745",          "PATO",  "electrical conductivity"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "pH",  "http://purl.obolibrary.org/obo/PATO_0001842",          "PATO",  "pH"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NH4", "http://purl.obolibrary.org/obo/CHEBI_28938",           "CHEBI", "ammonium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Na",  "http://purl.obolibrary.org/obo/CHEBI_29101",           "CHEBI", "sodium(1+)"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "K",   "http://purl.obolibrary.org/obo/CHEBI_29103",           "CHEBI", "potassium(1+)"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Ca",  "http://purl.obolibrary.org/obo/CHEBI_29108",           "CHEBI", "calcium(2+)"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Mg",  "http://purl.obolibrary.org/obo/CHEBI_18420",           "CHEBI", "magnesium(2+)"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cl",  "http://purl.obolibrary.org/obo/CHEBI_17996",           "CHEBI", "chloride"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NO3", "http://purl.obolibrary.org/obo/CHEBI_17632",           "CHEBI", "nitrate"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "SO4", "http://purl.obolibrary.org/obo/CHEBI_16189",           "CHEBI", "sulfate"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Pb",  "http://purl.obolibrary.org/obo/CHEBI_25016",           "CHEBI", "lead atom"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cd",  "http://purl.obolibrary.org/obo/CHEBI_22977",           "CHEBI", "cadmium atom"),

    # quality flags, flag values: 1=valid, 4=contaminated, 7=missing
    (TABLE_PRECIPITATION, precipitation_col_ids, "NS_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "LF_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "pH_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NH4_flag", "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Na_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "K_flag",   "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Ca_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Mg_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cl_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NO3_flag", "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "SO4_flag", "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Pb_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cd_flag",  "https://www.w3.org/ns/sosa/resultQuality",    "SOSA", "resultQuality"),

    # measurement_variables table
    (TABLE_VARIABLES, variables_col_ids, "variable_code", "https://www.w3.org/ns/sosa/ObservableProperty",  "SOSA", "ObservableProperty"),
    (TABLE_VARIABLES, variables_col_ids, "label",         "https://www.w3.org/ns/sosa/ObservableProperty",  "SOSA", "ObservableProperty"),
    (TABLE_VARIABLES, variables_col_ids, "unit",          "http://purl.obolibrary.org/obo/PATO_0000025",    "PATO", "unit of measurement"),
]

print("total mappings:", len(mappings))

total mappings: 34


In [19]:
# push every mapping to DBRepo

ok = 0
failed = 0

# for debugging
mappings = [
    # stations table
    (TABLE_STATIONS, stations_col_ids, "station_code", "https://www.w3.org/ns/sosa/Platform",             "SOSA",  "Platform"),
]

stations_col_ids      = get_column_ids(TABLE_STATIONS)

stations_col_ids

{'station_id': '9f762d20-696c-477f-8b74-7f7ba052f579',
 'station_code': '3ed0f861-e3ec-45e4-a48a-16fb8bf3852e',
 'latitude': '2e160049-5b95-4435-b490-8e2e628e4c30',
 'longitude': '0e0d51f7-55d3-4358-b355-dac60f65f8e1'}

In [20]:
type(stations_col_ids['station_code'])

str

In [26]:
for table_id, col_ids, column_name, uri, ontology, label in mappings:

    print('table_id == ', table_id, ' \n col_ids == ', col_ids, '\n column_name', column_name, '\n uri', uri, '\n ontology ' , ontology, '\n label', label)
    col_id = col_ids.get(column_name)
    if col_id is None:
        print(f"skip   {column_name:18s} (not found in table)")
        continue
    else:
        print ('FOUND in table ' , table_id, 'the column id: ' , col_id)
    try:
        print('database_id: ', DATABASE_ID,  ' - table_id: ', table_id , ' - column_id: ' , col_id , ' type col_id: ' , type(col_id) )
        col_id = str(col_id)
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=table_id,
            column_id=col_id,
            concept_uri=uri
        )
        #print(f"ok     {column_name:18s} ({ontology}: {label})")
        ok += 1
    except Exception as e:
        print(f"failed {column_name:18s} {str(e)[:80]}")
        failed += 1

 

print('THIS EXAMPLE ::: ')
print(DATABASE_ID , ' ' , table_id, ' ' , col_id )

print( col_id in stations_col_ids.values() )

client.update_table_column(
            database_id=DATABASE_ID,
            table_id=table_id,
            column_id=col_id,
            concept_uri=uri
        )
        
print(f"\n{ok} ok, {failed} failed")

table_id ==  53688cc1-5205-4f25-af30-7feef2ea1b2b  
 col_ids ==  {'station_id': '9f762d20-696c-477f-8b74-7f7ba052f579', 'station_code': '3ed0f861-e3ec-45e4-a48a-16fb8bf3852e', 'latitude': '2e160049-5b95-4435-b490-8e2e628e4c30', 'longitude': '0e0d51f7-55d3-4358-b355-dac60f65f8e1'} 
 column_name station_code 
 uri https://www.w3.org/ns/sosa/Platform 
 ontology  SOSA 
 label Platform
FOUND in table  53688cc1-5205-4f25-af30-7feef2ea1b2b the column id:  3ed0f861-e3ec-45e4-a48a-16fb8bf3852e
database_id:  bfa4385b-54a9-4ae3-b4f4-cb503d7bb016  - table_id:  53688cc1-5205-4f25-af30-7feef2ea1b2b  - column_id:  3ed0f861-e3ec-45e4-a48a-16fb8bf3852e  type col_id:  <class 'str'>
failed station_code       Failed to update column: not found
THIS EXAMPLE ::: 
bfa4385b-54a9-4ae3-b4f4-cb503d7bb016   53688cc1-5205-4f25-af30-7feef2ea1b2b   3ed0f861-e3ec-45e4-a48a-16fb8bf3852e
True


NotExistsError: Failed to update column: not found

In [ ]:
# verify the concepts are saved

for table_id, name in [(TABLE_STATIONS, "stations"), (TABLE_PRECIPITATION, "precipitation_measurements"), (TABLE_VARIABLES, "measurement_variables")]:
    print(f"\n{name}")
    table = client.get_table(database_id=DATABASE_ID, table_id=table_id)
    for col in table.columns:
        concept = getattr(col, "concept", None)
        uri = concept.uri if concept else "not set"
        print(f"  {col.name:20s} {uri}")


stations
  station_id           not set
  station_code         not set
  latitude             not set
  longitude            not set

precipitation_measurements
  measurement_id       not set
  station_id           not set
  sample_date          not set
  NS                   not set
  NS_flag              not set
  LF                   not set
  LF_flag              not set
  pH                   not set
  pH_flag              not set
  NH4                  not set
  NH4_flag             not set
  Na                   not set
  Na_flag              not set
  K                    not set
  K_flag               not set
  Ca                   not set
  Ca_flag              not set
  Mg                   not set
  Mg_flag              not set
  Cl                   not set
  Cl_flag              not set
  NO3                  not set
  NO3_flag             not set
  SO4                  not set
  SO4_flag             not set
  Pb                   not set
  Pb_flag              not set
 